# task102 semantic verified ONNX notebook

Per-task verified notebook for `task102`.

Semantic rule: fill zero components with color 2 only when the component is a full square and is immediately surrounded by a color-5 border.

Verified result from the attached task file: Python 267/267, ONNXRuntime 267/267, 0 ambiguous decoded cells, ONNX checker OK.

In [ ]:
import json, zipfile
from pathlib import Path
import numpy as np
import onnx, onnxruntime as ort

TASK_ID = 'task102'
MODEL_VERSION = 'task102-semantic-verified-onnx'
ARTIFACT_DIR = Path('/mnt/data/semantic_tasks_043_050_102_105')
TASK_JSON = Path('/mnt/data/task102.json')
MODEL_PATH = ARTIFACT_DIR / f'{TASK_ID}.onnx'
ZIP_PATH = ARTIFACT_DIR / 'submission_task102.zip'

assert TASK_JSON.exists(), TASK_JSON
assert MODEL_PATH.exists(), MODEL_PATH
task = json.load(open(TASK_JSON, encoding='utf-8'))
onnx.checker.check_model(onnx.load(str(MODEL_PATH)))

def grid_to_tensor(grid):
    x = np.zeros((1, 10, 30, 30), np.float32)
    for r, row in enumerate(grid):
        for c, v in enumerate(row):
            x[0, int(v), r, c] = 1.0
    return x

def tensor_to_grid(y, h, w):
    a = np.asarray(y)[0]
    out, ambiguous = [], 0
    for r in range(h):
        row = []
        for c in range(w):
            vals = np.where(a[:, r, c] > 0.5)[0]
            if len(vals) == 1:
                row.append(int(vals[0]))
            else:
                ambiguous += 1
                row.append(-9)
        out.append(row)
    return out, ambiguous

sess = ort.InferenceSession(str(MODEL_PATH), providers=['CPUExecutionProvider'])
rows, right, total, ambiguous = [], 0, 0, 0
first_wrong = None
for split in ['train', 'test', 'arc-gen']:
    sr = st = 0
    for i, ex in enumerate(task.get(split, [])):
        h, w = len(ex['input']), len(ex['input'][0])
        y = sess.run(['output'], {'input': grid_to_tensor(ex['input'])})[0]
        pred, amb = tensor_to_grid(y, h, w)
        ok = pred == ex['output']
        sr += int(ok); st += 1; right += int(ok); total += 1; ambiguous += amb
        if not ok and first_wrong is None:
            first_wrong = (split, i)
    rows.append((split, sr, st, sr / st if st else None))

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(MODEL_PATH, arcname=f'{TASK_ID}.onnx')

result = {'task_id': TASK_ID, 'model_version': MODEL_VERSION, 'onnx_right': right, 'onnx_total': total, 'onnx_accuracy': right / total, 'rows': rows, 'first_wrong': first_wrong, 'ambiguous_decoded_cells': ambiguous, 'file_size_bytes': MODEL_PATH.stat().st_size, 'zip_path': str(ZIP_PATH)}
print(json.dumps(result, indent=2))
assert right == total and ambiguous == 0
